In [ ]:
# STEP 1: Install required libraries

# torch → deep learning
# torchvision → image models
# pillow → image processing

!pip install torch torchvision pillow

In [ ]:
# STEP 2: Import required libraries

import torch
# torch → main deep learning library

import torchvision.transforms as transforms
# transforms → used to resize and convert images

from PIL import Image
# Image → to open and process images

import matplotlib.pyplot as plt
# matplotlib → to display images

import torchvision.models as models
# models → gives pre-trained models (like VGG19)

import requests

In [ ]:
# STEP 3: Load content and style images

import requests
from io import BytesIO

# Content image
content_url = "https://images.unsplash.com/photo-1503023345310-bd7c1de61c7d"

# Style image (use a reliable one)
style_url = "https://images.unsplash.com/photo-1549880338-65ddcdfd017b"

# Load content image
content_image = Image.open(BytesIO(requests.get(content_url).content)).convert("RGB")

# Load style image
style_image = Image.open(BytesIO(requests.get(style_url).content)).convert("RGB")

# Show images
plt.imshow(content_image)
plt.title("Content Image")
plt.axis("off")
plt.show()

plt.imshow(style_image)
plt.title("Style Image")
plt.axis("off")
plt.show()

In [ ]:
# STEP 4: Resize and transform images

# Define transformation
transform = transforms.Compose([
    transforms.Resize((256, 256)),   # resize image
    transforms.ToTensor()            # convert image → tensor
])

# Apply transformation
content = transform(content_image).unsqueeze(0)
style = transform(style_image).unsqueeze(0)

print("Images transformed successfully!")

In [ ]:
# STEP 5: Load pre-trained VGG19 model

# Load VGG19 model (pre-trained on ImageNet)
model = models.vgg19(pretrained=True).features

# Set model to evaluation mode (no training)
model.eval()

print("VGG19 model loaded successfully!")
# Freeze model parameters (VERY IMPORTANT)

for param in model.parameters():
    param.requires_grad = False

In [ ]:
# STEP 6: Function to extract features from image

def get_features(image, model):
    features = {}

    # Layers we are interested in
    layers = {
        '0': 'conv1',   # early layer (edges)
        '5': 'conv2',   # textures
        '10': 'conv3',
        '19': 'conv4',  # content layer (IMPORTANT)
        '28': 'conv5'
    }

    x = image

    # Pass image through model layer by layer
    for name, layer in model._modules.items():
        x = layer(x)

        # Save features at selected layers
        if name in layers:
            features[layers[name]] = x

    return features


# Extract features
content_features = get_features(content, model)
style_features = get_features(style, model)

print("Features extracted successfully!")

In [ ]:
# STEP 7: Generate styled image

# Clone content image
target = content.clone().requires_grad_(True)

# Freeze model (important)
for param in model.parameters():
    param.requires_grad = False

# Detach features (important)
content_features = {k: v.detach() for k, v in content_features.items()}
style_features = {k: v.detach() for k, v in style_features.items()}

# Optimizer
optimizer = torch.optim.Adam([target], lr=0.01)

# Training loop
for i in range(30):   # keep small

    target_features = get_features(target, model)

    # Content loss
    content_loss = torch.mean((target_features['conv4'] - content_features['conv4'])**2)

    # Style loss
    style_loss = 0
    for layer in style_features:
        style_loss += torch.mean((target_features[layer] - style_features[layer])**2)

    total_loss = content_loss + style_loss

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    if i % 10 == 0:
        print(f"Step {i}, Loss: {total_loss.item()}")

In [ ]:
# Convert tensor → image

output = target.detach().squeeze().cpu().numpy()
output = output.transpose(1, 2, 0)

# Clip values (VERY IMPORTANT)
output = output.clip(0, 1)

plt.imshow(output)
plt.title("Styled Image")
plt.axis("off")
plt.show()